<a href="https://colab.research.google.com/github/Nahom32/Cow-Anomaly-Detection/blob/main/notebooks/Cow_Localization_and_detection_version(s).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Dependencies

In [ ]:
!pip install ultralytics scipy torch torchvision pytorchvideo decord opencv-python tqdm pandas kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 21.7 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=17b510547eaefb3fdb81fd7a072753e5b9d0df9ea5448a9afac11f54b0cf1ab8
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=65bca54c24841168e0e3c9de4bf54293

### 2. Download Dataset from Kaggle

In [ ]:
import kagglehub

path = kagglehub.dataset_download("fandaoerji/cbvd-5cow-behavior-video-dataset")
print(path)
import os

for d in os.listdir(path):
    print(d)

100%|██████████| 10.8G/10.8G [01:32<00:00, 126MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11
labelframes
labelframes_add
CBVD-5.csv
miniannotations
videos_add
videos
annotations
rawframes_mini
minilabelframes


### 3. Import Libraries and Mount Drive

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import shutil
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit

# YOLO
from ultralytics import YOLO
from google.colab import drive, files
drive.mount('/content/drive')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


### 4. Configure Paths and Load Annotations

In [ ]:
DATA_ROOT = "/root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11"
ANNOTATIONS_DIR = os.path.join(DATA_ROOT, "annotations")      # full annotations
FRAMES_DIR = os.path.join(DATA_ROOT, "rawframes_mini")       # adjust to your frames folder
OUTPUT_DIR = "/content/cow_detection_dataset_fixed"
FPS = 25
TRAIN_RATIO = 0.8
RANDOM_SEED = 42
ann_file = os.path.join(ANNOTATIONS_DIR, "ava_train_v2.1.csv")
df = pd.read_csv(ann_file, header=None, dtype={0: str})
df.columns = ["video_id", "timestamp", "x1", "y1", "x2", "y2", "action_id", "target_id"]
print(f"Raw annotations: {len(df)}")
valid_video_ids = set(os.listdir(FRAMES_DIR))
df = df[df["video_id"].isin(valid_video_ids)].copy()
print(f"After filtering by existing video folders: {len(df)}")


Raw annotations: 34102
After filtering by existing video folders: 34025


### 5. Group and Deduplicate Bounding Boxes

In [ ]:
boxes_per_image = defaultdict(set)

for _, row in tqdm(df.iterrows(), total=len(df), desc="Grouping & dedup"):
    video_id = str(row["video_id"])
    frame_idx = int(row["timestamp"] * FPS)
    img_path = Path(FRAMES_DIR) / video_id / f"img_{frame_idx:05d}.jpg"

    if not img_path.exists():
        continue

    # Get normalized bounding box (already normalized in CSV)
    x1 = max(0.0, min(1.0, row["x1"]))
    x2 = max(0.0, min(1.0, row["x2"]))
    y1 = max(0.0, min(1.0, row["y1"]))
    y2 = max(0.0, min(1.0, row["y2"]))

    if x1 >= x2 or y1 >= y2:
        continue

    # Convert to YOLO format (center, width, height)
    x_center = (x1 + x2) / 2.0
    y_center = (y1 + y2) / 2.0
    width = x2 - x1
    height = y2 - y1

    # Use a tuple of 4 floats as the key for deduplication
    box_key = (round(x_center, 6), round(y_center, 6), round(width, 6), round(height, 6))

    image_key = f"{video_id}_{frame_idx:05d}"
    boxes_per_image[image_key].add( (str(img_path), box_key) )

print(f"Unique images with at least one cow: {len(boxes_per_image)}")

Grouping & dedup: 100%|██████████| 34025/34025 [00:03<00:00, 10301.28it/s]

Unique images with at least one cow: 2921


### 6. Split Data into Train and Validation Sets

In [ ]:
all_video_ids = list(set(key.split("_")[0] for key in boxes_per_image.keys()))
splitter = GroupShuffleSplit(n_splits=1, test_size=1-TRAIN_RATIO, random_state=RANDOM_SEED)
train_idx, val_idx = next(splitter.split(all_video_ids, groups=all_video_ids))

train_videos = set([all_video_ids[i] for i in train_idx])
val_videos = set([all_video_ids[i] for i in val_idx])
print(f"Train videos: {len(train_videos)}, Val videos: {len(val_videos)}")

Train videos: 392, Val videos: 99


### 7. Generate YOLO Format Dataset Files

In [ ]:
for image_key, box_set in tqdm(boxes_per_image.items(), desc="Writing dataset"):
    video_id = image_key.split("_")[0]
    split = "val" if video_id in val_videos else "train"

    # Get the image path (any element from the set, all have same path)
    img_path = next(iter(box_set))[0]

    # Copy image to destination
    dest_img = Path(OUTPUT_DIR) / split / "images" / f"{image_key}.jpg"
    if not dest_img.exists():
        shutil.copy(img_path, dest_img)

    # Write label file with one line per unique box
    label_path = Path(OUTPUT_DIR) / split / "labels" / f"{image_key}.txt"
    with open(label_path, "w") as f:
        for _, box_key in box_set:
            xc, yc, w, h = box_key
            f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

print(f"Dataset ready at {OUTPUT_DIR}")

Writing dataset: 100%|██████████| 2921/2921 [00:00<00:00, 5453.85it/s]

Dataset ready at /content/cow_detection_dataset_fixed


### 8. Train YOLO Model

In [ ]:
model = YOLO("yolo26m.pt")
results = model.train(
    data="/content/cow_detection_fixed.yaml",
    epochs=300,
    imgsz=1280,               # higher resolution if GPU memory allows (batch size may drop)
    batch=8,                  # adjust based on GPU memory (T4: 8 works for 1280px)
    patience=50,              # early stopping
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    box=7.5,                  # box loss gain
    cls=0.5,                  # class loss gain (only one class, but keep)
    dfl=1.5,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.2,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    optimizer="auto",         # will use AdamW or SGD, YOLO chooses well
    device=0,
    workers=4,
    project="cow_detector",
    name="yolov8m_cbvd_fixed",
    exist_ok=True,
    verbose=True
)


Ultralytics 8.4.49 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cow_detection_fixed.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.8, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_cbvd_fixed, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,